<a href="https://colab.research.google.com/github/adg1205/CSE425-Project/blob/master/Medium%20Task/Medium_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install librosa soundfile scikit-learn umap-learn matplotlib tqdm sentence-transformers

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
AUDIO_ROOT  = "/content/drive/MyDrive/Project Dataset/genres_original"
LYRICS_ROOT = "/content/drive/MyDrive/Project Dataset/lyrics_generated"

OUT_DIR     = "/content/drive/MyDrive/medium_task_results"
LATVIS_DIR  = os.path.join(OUT_DIR, "latent_visualization")
os.makedirs(LATVIS_DIR, exist_ok=True)

print("AUDIO_ROOT:", os.path.isdir(AUDIO_ROOT), AUDIO_ROOT)
print("LYRICS_ROOT:", os.path.isdir(LYRICS_ROOT), LYRICS_ROOT)
print("OUT_DIR:", OUT_DIR)

AUDIO_ROOT: True /content/drive/MyDrive/Project Dataset/genres_original
LYRICS_ROOT: True /content/drive/MyDrive/Project Dataset/lyrics_generated
OUT_DIR: /content/drive/MyDrive/medium_task_results


In [4]:
#Building paired index (audio+lyrics)
import glob
import pandas as pd

def build_paired_index(audio_root, lyrics_root):
    genre_dirs = sorted([d for d in glob.glob(os.path.join(audio_root, "*")) if os.path.isdir(d)])
    rows = []
    for gdir in genre_dirs:
        genre = os.path.basename(gdir)
        wavs = sorted(glob.glob(os.path.join(gdir, "*.wav")))
        for wav in wavs:
            track_id = os.path.splitext(os.path.basename(wav))[0]
            # Expected: /lyrics_generated/<genre>/<track_id>.txt (matches your earlier transcription layout)
            lpath = os.path.join(lyrics_root, genre, f"{track_id}.txt")
            rows.append((genre, track_id, wav, lpath if os.path.exists(lpath) else None))

    df = pd.DataFrame(rows, columns=["genre", "track_id", "audio_path", "lyrics_path"])
    coverage = df["lyrics_path"].notna().mean() * 100
    print(f"Tracks: {len(df)} | Genres: {df['genre'].nunique()} | Lyrics coverage: {coverage:.1f}%")
    return df

df = build_paired_index(AUDIO_ROOT, LYRICS_ROOT)
df.head()

Tracks: 700 | Genres: 7 | Lyrics coverage: 100.0%


,genre,track_id,audio_path,lyrics_path
0,blues,blues.00000,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
1,blues,blues.00001,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
2,blues,blues.00002,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
3,blues,blues.00003,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...
4,blues,blues.00004,/content/drive/MyDrive/Project Dataset/genres_...,/content/drive/MyDrive/Project Dataset/lyrics_...


In [5]:
#Audio Preprocessing
import numpy as np
import librosa

SR = 22050
DURATION = 30.0
N_MELS = 64
N_FFT = 1024
HOP = 512

TARGET_SAMPLES = int(SR * DURATION)
TARGET_FRAMES = 1 + TARGET_SAMPLES // HOP

def logmel_fixed(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    else:
        y = y[:TARGET_SAMPLES]

    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0
    )
    logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)  # (N_MELS, T)

    # Force fixed frames
    if logmel.shape[1] < TARGET_FRAMES:
        logmel = np.pad(logmel, ((0,0),(0, TARGET_FRAMES-logmel.shape[1])), mode="constant")
    else:
        logmel = logmel[:, :TARGET_FRAMES]
    return logmel

def estimate_audio_norm(df, max_items=300):
    idxs = np.arange(len(df))
    if len(df) > max_items:
        np.random.seed(42)
        idxs = np.random.choice(idxs, size=max_items, replace=False)
    mats = np.stack([logmel_fixed(df.loc[i, "audio_path"]) for i in idxs], axis=0)  # (B,M,T)
    return float(mats.mean()), float(mats.std() + 1e-6)

AUDIO_MEAN, AUDIO_STD = estimate_audio_norm(df, max_items=300)
AUDIO_MEAN, AUDIO_STD

(-38.717933654785156, 14.694233894348145)

In [6]:
#Lyrics Embeddings
from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "all-MiniLM-L6-v2"
text_model = SentenceTransformer(TEXT_MODEL_NAME)

def read_text(path):
    if path is None:
        return ""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

texts = [read_text(p) for p in df["lyrics_path"].tolist()]
E = text_model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
E = E.astype(np.float32)

# Save for reuse
np.save(os.path.join(OUT_DIR, "lyrics_embeddings.npy"), E)
df.to_csv(os.path.join(OUT_DIR, "paired_index.csv"), index=False)

print("Lyrics embeddings:", E.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Lyrics embeddings: (700, 384)


In [7]:
#PyTorch Dataset/DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

genres = sorted(df["genre"].unique().tolist())
genre_to_id = {g:i for i,g in enumerate(genres)}
Y = np.array([genre_to_id[g] for g in df["genre"]], dtype=np.int64)  # used for optional ARI [file:1]

class AudioLyricsDataset(Dataset):
    def __init__(self, df, E, y, audio_mean, audio_std):
        self.df = df.reset_index(drop=True)
        self.E = E
        self.y = y
        self.audio_mean = audio_mean
        self.audio_std = audio_std

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        x = logmel_fixed(self.df.loc[i, "audio_path"])
        x = (x - self.audio_mean) / self.audio_std
        x = torch.tensor(x[None, :, :], dtype=torch.float32)  # (1,M,T)

        e = torch.tensor(self.E[i], dtype=torch.float32)      # (D,)
        y = torch.tensor(self.y[i], dtype=torch.long)         # label for ARI (optional)
        return x, e, y

ds = AudioLyricsDataset(df, E, Y, AUDIO_MEAN, AUDIO_STD)
dl = DataLoader(ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [8]:
# Hybrid Convolutional VAE
import torch.nn as nn
import torch.nn.functional as F

class ConvEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

class ConvDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
        )
    def forward(self, h): return self.net(h)

class HybridConvVAE(nn.Module):
    def __init__(self, n_mels, t_frames, lyrics_dim, latent_dim=16):
        super().__init__()
        self.enc = ConvEncoder()

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_mels, t_frames)
            h = self.enc(dummy)
            self.h_shape = h.shape[1:]           # (C,H,W)
            self.h_dim = int(np.prod(self.h_shape))

        # audio branch
        self.audio_fc = nn.Linear(self.h_dim, 256)

        # lyrics branch
        self.lyrics_mlp = nn.Sequential(
            nn.Linear(lyrics_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )

        # fusion -> latent
        self.fuse = nn.Sequential(nn.Linear(256 + 128, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # decoder
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, self.h_dim), nn.ReLU(),
        )
        self.dec = ConvDecoder()

    def encode(self, x, e):
        h = self.enc(x).view(x.size(0), -1)
        a = torch.relu(self.audio_fc(h))
        l = self.lyrics_mlp(e)
        f = self.fuse(torch.cat([a, l], dim=1))
        return self.fc_mu(f), self.fc_logvar(f)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.dec_fc(z).view(z.size(0), *self.h_shape)
        return self.dec(h)

    def forward(self, x, e):
        mu, logvar = self.encode(x, e)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

def loss_fn(x, x_hat, mu, logvar, beta=1.0):
    # Crop in case transpose conv overshoots by a few frames
    x_hat = x_hat[:, :, :x.shape[2], :x.shape[3]]
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kld, recon, kld

In [9]:
#Train VAE and Extract latent vectors
import torch
import pandas as pd

vae = HybridConvVAE(n_mels=N_MELS, t_frames=TARGET_FRAMES, lyrics_dim=E.shape[1], latent_dim=16).to(DEVICE)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

EPOCHS = 30
BETA = 1.0

history = []
vae.train()
for ep in range(1, EPOCHS + 1):
    tot = rec = kld = 0.0
    n = 0
    for xb, eb, _ in dl:
        xb, eb = xb.to(DEVICE), eb.to(DEVICE)
        x_hat, mu, logvar = vae(xb, eb)
        loss, r, k = loss_fn(xb, x_hat, mu, logvar, beta=BETA)

        opt.zero_grad()
        loss.backward()
        opt.step()

        bs = xb.size(0)
        n += bs
        tot += loss.item() * bs
        rec += r.item() * bs
        kld += k.item() * bs

    history.append({"epoch": ep, "loss": tot/n, "recon": rec/n, "kld": kld/n})
    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:02d} | loss={tot/n:.4f} recon={rec/n:.4f} kld={kld/n:.4f}")

pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "vae_train_history.csv"), index=False)

# latent means (mu)
vae.eval()
Z = []
with torch.no_grad():
    for xb, eb, _ in DataLoader(ds, batch_size=32, shuffle=False):
        mu, _ = vae.encode(xb.to(DEVICE), eb.to(DEVICE))
        Z.append(mu.cpu().numpy())
Z = np.concatenate(Z, axis=0)
print("Z:", Z.shape)

Epoch 01 | loss=0.7757 recon=0.7715 kld=0.0042
Epoch 05 | loss=0.5080 recon=0.4614 kld=0.0465
Epoch 10 | loss=0.4945 recon=0.4440 kld=0.0504
Epoch 15 | loss=0.4773 recon=0.4226 kld=0.0547
Epoch 20 | loss=0.4687 recon=0.4172 kld=0.0514
Epoch 25 | loss=0.4727 recon=0.4194 kld=0.0533
Epoch 30 | loss=0.4687 recon=0.4126 kld=0.0561
Z: (700, 16)


In [10]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

def eval_metrics(X, labels, y_true=None):
    labels = np.asarray(labels)
    uniq = set(labels.tolist())
    uniq_no_noise = {u for u in uniq if u != -1}

    out = {"n_clusters": len(uniq_no_noise), "n_noise": int(np.sum(labels == -1))}
    if len(uniq_no_noise) >= 2:
        out["silhouette"] = float(silhouette_score(X, labels))
        out["davies_bouldin"] = float(davies_bouldin_score(X, labels))
    else:
        out["silhouette"] = np.nan
        out["davies_bouldin"] = np.nan

    out["ARI"] = float(adjusted_rand_score(y_true, labels)) if y_true is not None else np.nan
    return out

# ---------- Build hybrid features for PCA baseline ----------
def audio_pooled_logmel(df):
    feats = []
    for p in df["audio_path"].tolist():
        X = logmel_fixed(p)  # (N_MELS, T)
        feats.append(np.concatenate([X.mean(axis=1), X.std(axis=1)], axis=0))  # (2*N_MELS,)
    return np.stack(feats, axis=0).astype(np.float32)

A = audio_pooled_logmel(df)
L = E.astype(np.float32)

A_std = StandardScaler().fit_transform(A)
L_std = StandardScaler().fit_transform(L)
H = np.concatenate([A_std, L_std], axis=1)

k = df["genre"].nunique()
pca_dim = Z.shape[1]

H_pca = PCA(n_components=pca_dim, random_state=42).fit_transform(H)
H_pca_s = StandardScaler().fit_transform(H_pca)

# ---------- VAE latent standardized ----------
Zs = StandardScaler().fit_transform(Z)

rows = []

# 1) Baseline: PCA + KMeans (only baseline)
lab = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(H_pca_s)
rows.append({
    "method": "BASELINE",
    "embedding": f"PCA({pca_dim}) on [audio_pool + lyrics_embed]",
    "clusterer": "KMeans",
    **eval_metrics(H_pca_s, lab, y_true=Y)
})

# 2) VAE + KMeans
lab = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(Zs)
rows.append({
    "method": "VAE",
    "embedding": f"latent_dim={Z.shape[1]}",
    "clusterer": "KMeans",
    **eval_metrics(Zs, lab, y_true=Y)
})

# 3) VAE + Agglomerative
lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Zs)
rows.append({
    "method": "VAE",
    "embedding": f"latent_dim={Z.shape[1]}",
    "clusterer": "Agglomerative(ward)",
    **eval_metrics(Zs, lab, y_true=Y)
})

# 4) VAE + DBSCAN (tiny eps sweep -> pick best; INCLUDE in final CSV)
eps_grid = [0.3, 0.5, 0.8, 1.0]
min_samples = 10

sweep_rows = []
for eps in eps_grid:
    lab = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Zs)  # noise label = -1 [web:254]
    sweep_rows.append({
        "method": "VAE",
        "embedding": f"latent_dim={Z.shape[1]}",
        "clusterer": f"DBSCAN(eps={eps},min={min_samples})",
        **eval_metrics(Zs, lab, y_true=Y)
    })

dbscan_sweep_df = pd.DataFrame(sweep_rows)

# Select best eps by highest silhouette (fallback: first row if all NaN)
tmp = dbscan_sweep_df.dropna(subset=["silhouette"])
best_dbscan = tmp.sort_values("silhouette", ascending=False).head(1) if len(tmp) else dbscan_sweep_df.head(1)

rows.append(best_dbscan.iloc[0].to_dict())  # <-- this ensures DBSCAN is inside the final CSV

# ---------- Save one final CSV that includes DBSCAN ----------
metrics_df = pd.DataFrame(rows)
final_path = os.path.join(OUT_DIR, "clustering_metrics_medium_final.csv")
metrics_df.to_csv(final_path, index=False)

# (optional) save the sweep for transparency/debugging
sweep_path = os.path.join(OUT_DIR, "dbscan_eps_sweep.csv")
dbscan_sweep_df.to_csv(sweep_path, index=False)

print("Saved final metrics (includes DBSCAN):", final_path)
print("Saved DBSCAN sweep:", sweep_path)

metrics_df

Saved final metrics (includes DBSCAN): /content/drive/MyDrive/medium_task_results/clustering_metrics_medium_final.csv
Saved DBSCAN sweep: /content/drive/MyDrive/medium_task_results/dbscan_eps_sweep.csv


,method,embedding,clusterer,n_clusters,n_noise,silhouette,davies_bouldin,ARI
0,BASELINE,PCA(16) on [audio_pool + lyrics_embed],KMeans,7,0,0.100293,2.611457,0.079522
1,VAE,latent_dim=16,KMeans,7,0,0.359767,0.936835,0.170625
2,VAE,latent_dim=16,Agglomerative(ward),7,0,0.364445,0.888918,0.151534
3,VAE,latent_dim=16,"DBSCAN(eps=0.8,min=10)",4,198,0.112877,1.823400,0.041001


In [11]:
#Visualization (UMAP)
import umap
import matplotlib.pyplot as plt

um = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
Z2 = um.fit_transform(Zs)

# pick one set of labels to color (e.g., KMeans)
labels = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(Zs)

plt.figure(figsize=(7,6))
plt.scatter(Z2[:,0], Z2[:,1], c=labels, s=10, cmap="tab10")
plt.title("UMAP: Hybrid Conv-VAE latent (colored by KMeans clusters)")
plt.tight_layout()
plt.savefig(os.path.join(LATVIS_DIR, "umap_latent_kmeans.png"), dpi=200)
plt.close()


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
